# Tamreena AI — Agent Exploration Notebook

This notebook explores and validates the full multi-agent pipeline **before** moving to production Python files.

**RAG is hardcoded** in this notebook — the RAG team will wire in Pinecone + embeddings later. Everything else (agents, tools, memory, MongoDB, FastAPI) is real.

**v1.3 update:** the Supervisor and Exercise Recommender now classify the user's free-text goal into one of 7 programming paradigms (hypertrophy / strength / fat_loss / general_fitness / athletic_performance / endurance_complement / rehabilitation) before planning begins. Every downstream decision — intensity zone labels, DAY MAP time-per-set, rep/rest/RPE tables — is paradigm-conditional. See `tamrena_architecture_2.md` Section 4e and the v1.3 changelog entry (Section 14) for the full design.

### Build order
1. Environment setup & imports
2. Shared MD memory tools
3. InBody parser (VLM tool)
4. Exercise database (MongoDB) + hardcoded RAG stub
5. Supervisor agent (goal → paradigm classification, DAY MAP, progress tracking)
6. Exercise Recommender sub-agent (paradigm-conditional intensity tables)
7. Plan Assembler sub-agent
8. End-to-end pipeline run — tests two different paradigms


## Part 1 — Environment Setup & Imports

Load `.env` credentials, initialize the `AzureChatOpenAI` instance that every agent will share, and verify the connection is live before touching anything else.

In [29]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv(dotenv_path=os.path.join("..", ".env"), override=True)

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0.3,
    timeout=60,
    max_retries=2
)

# quick sanity check
resp = llm.invoke("Say 'Tamreena online' and nothing else.")
print(resp.content)

Tamreena online


## Part 2 — Shared MD Memory Tools + Progress Tracking

All agents communicate through a single markdown file per session.
- `read_plan_memory(session_id)` — any agent calls this first to get full context
- `write_plan_memory(session_id, section_title, content)` — append-only, never overwrites
- `validate_session_duration(session_id)` — Plan Assembler calls this after scheduling to confirm no day exceeds its DAY MAP set budget

Sequential execution means no concurrent write conflicts.

**Progress tracking (separate from the MD file):** an earlier test run silently dropped an entire muscle group (back) — the Supervisor's own dispatch loop lost track of which muscle groups were actually done, and nothing caught it before the final plan was returned. `progress.json` fixes this with structured state, not text parsing:
- `init_plan_progress(session_id, muscle_groups)` — Supervisor calls once, right after computing the DAY MAP, with the authoritative list of muscle groups this plan requires
- `mark_step_done(session_id, muscle_group)` — Exercise Recommender calls as its last action
- `get_plan_progress(session_id)` — Supervisor calls after every dispatch to cross-check, not just trust its own memory
- `validate_plan_completeness(session_id)` — Plan Assembler calls first; refuses to assemble a plan around a missing muscle group

In [30]:
import os
import re
import json
from langchain_core.tools import tool

SESSION_DIR = os.path.join("..", "sessions")

def _plan_path(session_id: str) -> str:
    return os.path.join(SESSION_DIR, session_id, "plan.md")

def _progress_path(session_id: str) -> str:
    return os.path.join(SESSION_DIR, session_id, "progress.json")

@tool
def read_plan_memory(session_id: str) -> str:
    """Read the full shared plan memory file for this session. Call this first before doing any work."""
    path = _plan_path(session_id)
    if not os.path.exists(path):
        return "(plan file not created yet)"
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

@tool
def write_plan_memory(session_id: str, section_title: str, content: str) -> str:
    """Append a completed section to the shared plan memory file. Never call this more than once per section."""
    path = _plan_path(session_id)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(f"\n\n## {section_title}\n{content}\n\n---")
    return f"Written to plan memory: {section_title}"


@tool
def validate_session_duration(session_id: str) -> str:
    """
    Plan Assembler calls this AFTER writing the Weekly Schedule. Parses the DAY MAP's
    max_sets budgets and the scheduled sets per day from the Weekly Schedule section,
    and returns PASS or the specific days/set-counts that are over budget.
    """
    path = _plan_path(session_id)
    if not os.path.exists(path):
        return "NO PLAN FILE FOUND — cannot validate."
    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    # parse DAY MAP lines: "Day 1 — Legs: muscles [...] | max_sets: 26 | intensity: medium"
    day_budgets = {}
    for line in content.splitlines():
        stripped = line.strip()
        if stripped.startswith("Day ") and "max_sets:" in stripped:
            day_label = stripped.split("—")[0].strip()
            try:
                max_sets = int(stripped.split("max_sets:")[1].split("|")[0].strip())
                day_budgets[day_label] = max_sets
            except (ValueError, IndexError):
                continue

    if not day_budgets:
        return "NO DAY MAP FOUND — cannot validate. Supervisor must write DAY MAP before dispatching agents."

    # count scheduled sets per day from "### Day N — ..." sections + "N×M" table cells
    day_set_counts = {}
    current_day = None
    for line in content.splitlines():
        stripped = line.strip()
        if stripped.startswith("### Day"):
            current_day = stripped.split("—")[0].replace("###", "").strip()
            day_set_counts[current_day] = 0
        elif current_day and stripped.startswith("|"):
            for part in stripped.split("|"):
                match = re.search(r"(\d+)\s*[×xX]\s*\d+", part.strip())
                if match:
                    day_set_counts[current_day] += int(match.group(1))

    violations = []
    for day, budget in day_budgets.items():
        if day not in day_set_counts:
            violations.append(f"  {day}: no schedule table found for this day — cannot verify budget")
            continue
        scheduled = day_set_counts[day]
        if scheduled > budget:
            violations.append(f"  {day}: {scheduled} sets scheduled, budget is {budget} ({scheduled - budget} over — trim)")

    if not violations:
        return "PASS — all days within session duration budget."
    return "VIOLATIONS — trim before writing final plan:\n" + "\n".join(violations)


@tool
def init_plan_progress(session_id: str, muscle_groups: list[str]) -> str:
    """Supervisor calls this ONCE, right after deciding the split and writing the DAY MAP —
    before dispatching any exercise-recommender. muscle_groups must be the complete,
    authoritative list of muscle-group ids this plan requires (e.g. including separate
    ids like 'legs_a'/'legs_b' when there are two leg days)."""
    path = _progress_path(session_id)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump({"expected": muscle_groups, "completed": []}, f)
    return f"Progress tracker initialized: {len(muscle_groups)} muscle groups expected: {muscle_groups}"


@tool
def mark_step_done(session_id: str, muscle_group: str) -> str:
    """Exercise-recommender calls this as its LAST action, immediately after write_plan_memory,
    using the exact muscle_group id it was given in its task prompt."""
    path = _progress_path(session_id)
    if not os.path.exists(path):
        return "WARNING: progress tracker not initialized for this session."
    with open(path, "r", encoding="utf-8") as f:
        progress = json.load(f)

    if muscle_group not in progress["expected"]:
        return f"WARNING: '{muscle_group}' is not in the expected list {progress['expected']}. Not recorded."
    if muscle_group in progress["completed"]:
        return f"WARNING: '{muscle_group}' was already marked done — possible duplicate dispatch."

    progress["completed"].append(muscle_group)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(progress, f)

    remaining = [g for g in progress["expected"] if g not in progress["completed"]]
    return f"Marked done: {muscle_group}. Remaining: {remaining or 'none — all groups complete'}"


@tool
def get_plan_progress(session_id: str) -> str:
    """Supervisor calls this after EVERY task() return, to confirm the muscle group just
    dispatched actually got marked done before deciding what to dispatch next. Also call
    before dispatching the plan-assembler — only proceed if all_done is true."""
    path = _progress_path(session_id)
    if not os.path.exists(path):
        return json.dumps({"error": "progress tracker not initialized"})
    with open(path, "r", encoding="utf-8") as f:
        progress = json.load(f)
    remaining = [g for g in progress["expected"] if g not in progress["completed"]]
    return json.dumps({
        "expected": progress["expected"],
        "completed": progress["completed"],
        "remaining": remaining,
        "next": remaining[0] if remaining else None,
        "all_done": len(remaining) == 0,
    })


@tool
def validate_plan_completeness(session_id: str) -> str:
    """Plan Assembler calls this FIRST, before doing any scheduling work. Returns PASS only
    if every expected muscle group has been marked done."""
    result = json.loads(get_plan_progress.invoke({"session_id": session_id}))
    if result.get("all_done"):
        return "PASS — all expected muscle groups have prescriptions."
    return f"INCOMPLETE — missing prescriptions for: {result.get('remaining')}. Do not assemble the plan."


# --- test ---
import uuid
TEST_SESSION = str(uuid.uuid4())
print(write_plan_memory.invoke({"session_id": TEST_SESSION, "section_title": "Test Section", "content": "hello from test"}))
print(read_plan_memory.invoke({"session_id": TEST_SESSION}))
print(init_plan_progress.invoke({"session_id": TEST_SESSION, "muscle_groups": ["chest", "back"]}))
print(mark_step_done.invoke({"session_id": TEST_SESSION, "muscle_group": "chest"}))
print(get_plan_progress.invoke({"session_id": TEST_SESSION}))
print(validate_plan_completeness.invoke({"session_id": TEST_SESSION}))


Written to plan memory: Test Section


## Test Section
hello from test

---
Progress tracker initialized: 2 muscle groups expected: ['chest', 'back']
Marked done: chest. Remaining: ['back']
{"expected": ["chest", "back"], "completed": ["chest"], "remaining": ["back"], "next": "back", "all_done": false}
INCOMPLETE — missing prescriptions for: ['back']. Do not assemble the plan.


## Part 3 — InBody Parser (VLM Tool)

This is a tool (not a sub-agent) called by the supervisor as its first action.  
It sends the raw InBody scan image/PDF to Azure gpt-4.1-mini with vision and returns:
- Structured fitness data (SMM, BF%, segmental lean mass)
- **Flags** — the most important output, tells sub-agents what to do differently (asymmetries, elevated BF%, etc.)

In this notebook we also define a `parse_inbody_text` fallback for when you pass raw text instead of an image (useful during development).

In [31]:
import base64
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

INBODY_SYSTEM_PROMPT = """You are an InBody scan analysis tool. Extract fitness-relevant data and output structured text.

Output format (follow exactly):

INBODY ANALYSIS
───────────────
Skeletal Muscle Mass : {value} kg
Body Fat %           : {value}%
BMR                  : {value} kcal
Segmental Lean Mass:
  Right arm : {value} kg | Left arm : {value} kg  → Arm asymmetry: YES/NO (diff: {value}g)
  Right leg : {value} kg | Left leg : {value} kg  → Leg asymmetry: YES/NO (diff: {value}g)
  Trunk     : {value} kg

FLAGS (used by all sub-agents)
───────────────────────────────
ARM_ASYMMETRY   : YES/NO  — if YES, pull day must include unilateral movements
LEG_ASYMMETRY   : YES/NO  — if YES, leg day prioritises unilateral, start on weaker side
ELEVATED_BF     : YES/NO  — if YES (>18% male/>25% female), lean toward 12-15 rep ranges
TRUNK_UNDERDEVELOPED : YES/NO — if YES, chest/back volume gets priority

Extract what you can from the scan. If a value is not visible, write UNKNOWN."""


@tool
def parse_inbody(session_id: str, inbody_b64: str, content_type: str) -> str:
    """Parse an InBody scan image or PDF (base64 encoded) and return structured fitness data with flags."""
    msg = HumanMessage(content=[
        {
            "type": "image_url",
            "image_url": {"url": f"data:{content_type};base64,{inbody_b64}"},
        },
        {"type": "text", "text": INBODY_SYSTEM_PROMPT},
    ])
    result = llm.invoke([msg])
    return result.content


@tool
def parse_inbody_text(session_id: str, raw_text: str) -> str:
    """Parse InBody data from raw text (use when image is not available during dev/testing)."""
    prompt = f"""{INBODY_SYSTEM_PROMPT}

Raw InBody text to parse:
{raw_text}"""
    result = llm.invoke(prompt)
    return result.content


# --- test with a realistic fake InBody text ---
SAMPLE_INBODY_TEXT = """
InBody 570 Result Sheet
Name: Ahmed Al-Rashidi   Age: 27   Gender: Male

Body Composition Analysis
  Weight: 84.3 kg
  Skeletal Muscle Mass: 38.2 kg
  Body Fat Mass: 18.6 kg
  Body Fat %: 22.1%
  BMR: 1910 kcal

Segmental Lean Analysis (kg)
  Right Arm: 3.82   Left Arm: 3.41
  Right Leg: 10.95  Left Leg: 11.02
  Trunk: 28.7
"""

parsed = parse_inbody_text.invoke({"session_id": "test", "raw_text": SAMPLE_INBODY_TEXT})
print(parsed)

INBODY ANALYSIS
───────────────
Skeletal Muscle Mass : 38.2 kg
Body Fat %           : 22.1%
BMR                  : 1910 kcal
Segmental Lean Mass:
  Right arm : 3.82 kg | Left arm : 3.41 kg  → Arm asymmetry: YES (diff: 410g)
  Right leg : 10.95 kg | Left leg : 11.02 kg  → Leg asymmetry: NO (diff: 70g)
  Trunk     : 28.7 kg

FLAGS (used by all sub-agents)
───────────────────────────────
ARM_ASYMMETRY   : YES  — if YES, pull day must include unilateral movements
LEG_ASYMMETRY   : NO   — if YES, leg day prioritises unilateral, start on weaker side
ELEVATED_BF     : YES  — if YES (>18% male/>25% female), lean toward 12-15 rep ranges
TRUNK_UNDERDEVELOPED : NO — if YES, chest/back volume gets priority


## Part 4 — Exercise Database (MongoDB) + Hardcoded RAG Stub

### 4a — MongoDB exercise tool
Queries the `tamrena.exercises` collection. The `search_exercise_db` tool filters by muscle group, movement type, and contraindications.

### 4b — Hardcoded RAG stub (replaces Pinecone + embeddings)
`search_rag` returns curated, static training principles and exercise notes per muscle group.  
**This is a placeholder** — the RAG team will swap this function for the real Pinecone hybrid search without changing any other code.

In [32]:
# ── Hardcoded RAG stub ────────────────────────────────────────────────────────
# Principles that apply to every muscle group
_PRINCIPLES = """
[PRINCIPLE] Progressive overload is the primary driver of hypertrophy and strength. Increase load or reps weekly.
[PRINCIPLE] For hypertrophy, train each muscle group 10-20 sets/week across 2+ sessions for optimal frequency.
[PRINCIPLE] Rep ranges: strength 1-5, hypertrophy 6-15, endurance 15+. All ranges build muscle if taken near failure.
[PRINCIPLE] Rest periods: heavy compound 2-3 min, moderate 90s, isolation/corrective 60s.
[PRINCIPLE] RPE (Rate of Perceived Exertion) 8-9 = 1-2 reps in reserve. RPE 5-6 = comfortable, corrective work.
[PRINCIPLE] Elevated BF% (>18% male, >25% female): lean toward higher rep ranges (12-15) for better fat oxidation.
[PRINCIPLE] Asymmetry correction: always start unilateral sets on the weaker side. Never let the stronger side compensate.
[PRINCIPLE] Beginners: 10-12 sets/week per group. Intermediates: 14-18. Advanced: 18-22.
"""

# Muscle-specific notes (what the agent needs to make good exercise choices)
_MUSCLE_NOTES = {
    "chest": """
[CHEST] Chest has two primary functions: horizontal adduction (pressing) and shoulder flexion.
[CHEST] Compound first: flat or incline barbell/dumbbell press for maximum motor unit recruitment.
[CHEST] Incline angle (30-45°) shifts emphasis to clavicular head (upper chest) — prioritise for underdeveloped upper chest.
[CHEST] Cable flyes and pec deck provide constant tension through the stretched position — superior for hypertrophy isolation.
[CHEST] Dips (forward lean) are an effective compound chest movement when bars allow anterior tilt.
""",
    "back_vertical": """
[BACK-VERTICAL] Vertical pulling targets lats and teres major — responsible for the V-taper look.
[BACK-VERTICAL] Pull-ups and lat pulldowns are primary movements. Use full ROM — initiate with scapular depression.
[BACK-VERTICAL] Supinated grip (chin-up) increases bicep contribution and is easier for beginners.
[BACK-VERTICAL] Single-arm cable pulldown — best unilateral lat movement when arm asymmetry is flagged.
[BACK-VERTICAL] Straight-arm pulldown isolates lats without bicep contribution — good finisher.
""",
    "back_horizontal": """
[BACK-HORIZONTAL] Horizontal pulling targets mid/upper traps, rhomboids, and rear delts.
[BACK-HORIZONTAL] Barbell rows allow heaviest loading. Dumbbell rows allow greater ROM and unilateral correction.
[BACK-HORIZONTAL] Cable rows provide constant tension — use for hypertrophy finishers after heavy rows.
[BACK-HORIZONTAL] Chest-supported rows eliminate lower back fatigue — useful when lower back is a limiting factor.
[BACK-HORIZONTAL] Pronated grip rows emphasise rhomboids/traps. Neutral grip shifts more to lats.
""",
    "shoulders": """
[SHOULDERS] Overhead press (barbell or dumbbell) is the primary compound movement for overall shoulder development.
[SHOULDERS] Lateral raises are essential for medial deltoid — cannot be replaced by pressing movements.
[SHOULDERS] Arnold press adds rotation through ROM — hits all three heads. More fatiguing than standard press.
[SHOULDERS] Cable lateral raises maintain tension at the top — superior to dumbbell for hypertrophy.
[SHOULDERS] Seated press reduces core demand and allows heavier loading with better stability.
""",
    "rear_delts": """
[REAR-DELTS] Rear delts are typically undertrained. They require dedicated isolation work.
[REAR-DELTS] Face pulls with external rotation hit rear delts + rotator cuff — important for shoulder health.
[REAR-DELTS] Reverse pec deck and bent-over lateral raises are primary isolation movements.
[REAR-DELTS] Rear delts respond well to higher rep ranges (15-20) and high frequency.
[REAR-DELTS] Include in pull day to avoid shoulder imbalance from pressing volume.
""",
    "triceps": """
[TRICEPS] Triceps make up ~2/3 of upper arm mass — prioritise over biceps for arm size.
[TRICEPS] Long head (largest) is best targeted with overhead extension — requires full shoulder flexion.
[TRICEPS] Pushdowns target lateral and medial head — useful finisher but not sufficient alone.
[TRICEPS] Close-grip bench press is the heaviest tricep compound movement.
[TRICEPS] Dips (upright torso) emphasise triceps over chest — effective mass builder.
""",
    "biceps": """
[BICEPS] Biceps function: elbow flexion + supination. Both must be trained for full development.
[BICEPS] Barbell curl allows maximum load — key strength movement.
[BICEPS] Incline dumbbell curl stretches the long head at the bottom — superior for peak development.
[BICEPS] Hammer curl targets brachialis and brachioradialis — adds thickness to the arm.
[BICEPS] Cable curls maintain tension through full ROM — excellent for hypertrophy finisher.
""",
    "quads": """
[QUADS] Squats and leg press are primary compound movements. Squat depth matters — full depth maximises quad stretch.
[QUADS] Hack squat and leg press allow more knee-dominant pattern than back squat — better quad isolation.
[QUADS] Leg extension isolates quads without hip flexor contribution — critical for complete development.
[QUADS] Bulgarian split squat is the best unilateral quad movement — use when leg asymmetry is flagged.
[QUADS] Knee-over-toe training is safe and desirable for quad development when progressed correctly.
""",
    "hamstrings": """
[HAMSTRINGS] Hamstrings cross both hip and knee — require both hip hinge and knee flexion movements.
[HAMSTRINGS] Romanian deadlift (RDL) is the primary hip hinge — maximises hamstring length under load.
[HAMSTRINGS] Leg curl (lying or seated) is the primary knee flexion movement — do not skip it.
[HAMSTRINGS] Nordic hamstring curl builds extraordinary strength and injury resilience. Very high difficulty.
[HAMSTRINGS] Single-leg RDL is the unilateral choice when leg asymmetry is flagged.
""",
    "glutes": """
[GLUTES] Hip thrust is the most effective glute exercise — maximises glute activation at full hip extension.
[GLUTES] Romanian deadlift hits glutes and hamstrings together — good compound for posterior chain.
[GLUTES] Bulgarian split squat with forward lean shifts more load to glutes vs quads.
[GLUTES] Cable kickbacks and abductions isolate glute medius and minimus — needed for complete development.
[GLUTES] Glutes respond to both heavy loading (3-6 reps) and high rep ranges (15-20).
""",
    "calves": """
[CALVES] Calves have two muscles: gastrocnemius (knee extended) and soleus (knee bent).
[CALVES] Standing calf raise targets gastrocnemius. Seated calf raise targets soleus. Train both.
[CALVES] Calves require high volume (15-20 sets/week) and high reps (15-25) due to fibre type composition.
[CALVES] Full ROM critical — complete stretch at bottom, full plantarflexion at top.
[CALVES] Single-leg calf raises add unilateral correction when asymmetry is present.
""",
    "core": """
[CORE] Core training targets stability (anti-rotation, anti-extension) and flexion/rotation strength.
[CORE] Plank and dead bug are anti-extension standards — build spine-safe stability.
[CORE] Cable woodchops and Pallof press target rotational stability.
[CORE] Hanging leg raises and ab wheel rollouts are advanced loaded movements.
[CORE] Core work placed at end of session — pre-fatiguing core impairs compound lifts.
""",
}

@tool
def search_rag(muscle_group: str, query: str) -> str:
    """
    Search the RAG knowledge base for training principles and muscle-specific guidance.
    Returns relevant evidence to inform exercise selection and prescription.
    NOTE: This is a hardcoded stub — RAG team will replace with Pinecone hybrid search.
    """
    muscle_notes = _MUSCLE_NOTES.get(muscle_group, f"[No specific notes for {muscle_group}]")
    return f"=== PRINCIPLES ===\n{_PRINCIPLES}\n=== {muscle_group.upper()} SPECIFIC ===\n{muscle_notes}"


# --- test ---
print(search_rag.invoke({"muscle_group": "chest", "query": "hypertrophy chest exercises"}))
print("---")
print(search_exercise_db.invoke({"muscle_group": "chest", "movement_type": "compound"}))

=== PRINCIPLES ===

[PRINCIPLE] Progressive overload is the primary driver of hypertrophy and strength. Increase load or reps weekly.
[PRINCIPLE] For hypertrophy, train each muscle group 10-20 sets/week across 2+ sessions for optimal frequency.
[PRINCIPLE] Rep ranges: strength 1-5, hypertrophy 6-15, endurance 15+. All ranges build muscle if taken near failure.
[PRINCIPLE] Rest periods: heavy compound 2-3 min, moderate 90s, isolation/corrective 60s.
[PRINCIPLE] RPE (Rate of Perceived Exertion) 8-9 = 1-2 reps in reserve. RPE 5-6 = comfortable, corrective work.
[PRINCIPLE] Elevated BF% (>18% male, >25% female): lean toward higher rep ranges (12-15) for better fat oxidation.
[PRINCIPLE] Asymmetry correction: always start unilateral sets on the weaker side. Never let the stronger side compensate.
[PRINCIPLE] Beginners: 10-12 sets/week per group. Intermediates: 14-18. Advanced: 18-22.

=== CHEST SPECIFIC ===

[CHEST] Chest has two primary functions: horizontal adduction (pressing) and should

In [33]:
import sqlite3
from langchain_core.tools import tool
from typing import Optional

# ── SQLite exercise database (temporary stand-in until MongoDB is wired up) ──
DB_PATH = os.path.join("..", "data", "tamreena.db")
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

def get_db_connection():
    return sqlite3.connect(DB_PATH)

SEED_EXERCISES = [
    # (name, primary_muscle, movement_type, equipment, difficulty, contraindications)
    ("Flat Barbell Bench Press", "chest", "compound", "barbell", "intermediate", None),
    ("Incline Dumbbell Press", "chest", "compound", "dumbbell", "beginner", None),
    ("Cable Fly", "chest", "isolation", "cable", "beginner", None),
    ("Pec Deck", "chest", "isolation", "machine", "beginner", None),
    ("Weighted Dip", "chest", "compound", "bodyweight", "advanced", "shoulder_pain"),

    ("Pull-Up", "back", "compound", "bodyweight", "intermediate", None),
    ("Lat Pulldown", "back", "compound", "cable", "beginner", None),
    ("Barbell Row", "back", "compound", "barbell", "intermediate", "lower_back_pain"),
    ("Single-Arm Dumbbell Row", "back", "unilateral", "dumbbell", "beginner", None),
    ("Chest-Supported Row", "back", "compound", "machine", "beginner", "lower_back_pain"),
    ("Straight-Arm Pulldown", "back", "isolation", "cable", "beginner", None),

    ("Seated Dumbbell Overhead Press", "shoulders", "compound", "dumbbell", "beginner", None),
    ("Barbell Overhead Press", "shoulders", "compound", "barbell", "intermediate", "shoulder_pain"),
    ("Cable Lateral Raise", "shoulders", "isolation", "cable", "beginner", None),
    ("Dumbbell Lateral Raise", "shoulders", "isolation", "dumbbell", "beginner", None),
    ("Face Pull", "shoulders", "isolation", "cable", "beginner", None),
    ("Reverse Pec Deck", "shoulders", "isolation", "machine", "beginner", None),

    ("Barbell Curl", "arms", "isolation", "barbell", "beginner", None),
    ("Incline Dumbbell Curl", "arms", "isolation", "dumbbell", "beginner", None),
    ("Hammer Curl", "arms", "isolation", "dumbbell", "beginner", None),
    ("Close-Grip Bench Press", "arms", "compound", "barbell", "intermediate", None),
    ("Tricep Pushdown", "arms", "isolation", "cable", "beginner", None),
    ("Single-Arm Cable Curl", "arms", "unilateral", "cable", "beginner", None),

    ("Barbell Back Squat", "legs", "compound", "barbell", "intermediate", "knee_pain"),
    ("Leg Press", "legs", "compound", "machine", "beginner", "knee_pain"),
    ("Romanian Deadlift", "legs", "compound", "barbell", "intermediate", "lower_back_pain"),
    ("Leg Extension", "legs", "isolation", "machine", "beginner", "knee_pain"),
    ("Lying Leg Curl", "legs", "isolation", "machine", "beginner", None),
    ("Bulgarian Split Squat", "legs", "unilateral", "dumbbell", "intermediate", "knee_pain"),
    ("Standing Calf Raise", "legs", "isolation", "machine", "beginner", None),
    ("Hip Thrust", "legs", "compound", "barbell", "intermediate", None),
]

def init_exercise_db():
    conn = get_db_connection()
    conn.execute("""
        CREATE TABLE IF NOT EXISTS exercises (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            primary_muscle TEXT NOT NULL,
            movement_type TEXT NOT NULL,
            equipment TEXT,
            difficulty TEXT,
            contraindications TEXT
        )
    """)
    count = conn.execute("SELECT COUNT(*) FROM exercises").fetchone()[0]
    if count == 0:
        conn.executemany(
            "INSERT INTO exercises (name, primary_muscle, movement_type, equipment, difficulty, contraindications) VALUES (?, ?, ?, ?, ?, ?)",
            SEED_EXERCISES,
        )
        conn.commit()
    conn.close()

init_exercise_db()


@tool
def search_exercise_db(
    muscle_group: str,
    movement_type: str = "all",
    exclude_contraindication: Optional[str] = None,
) -> str:
    """
    Query the exercise database for exercises matching a muscle group.
    movement_type: compound | isolation | unilateral | all
    exclude_contraindication: body part to avoid (e.g. 'knee_pain')
    """
    conn = get_db_connection()
    query = "SELECT name, equipment, difficulty, movement_type, contraindications FROM exercises WHERE primary_muscle = ?"
    params = [muscle_group]
    if movement_type != "all":
        query += " AND movement_type = ?"
        params.append(movement_type)
    rows = conn.execute(query, params).fetchall()
    conn.close()

    if exclude_contraindication:
        rows = [r for r in rows if not (r[4] and exclude_contraindication in r[4])]

    if not rows:
        return f"No exercises found for [{muscle_group}] [{movement_type}]"

    lines = [f"  • {r[0]} ({r[1] or '?'}, {r[2] or '?'}, {r[3] or '?'})" for r in rows]
    return f"DB results — muscle: [{muscle_group}] | type: [{movement_type}]\n" + "\n".join(lines)


# --- test ---
print(search_exercise_db.invoke({"muscle_group": "chest", "movement_type": "compound"}))
print("---")
print(search_exercise_db.invoke({"muscle_group": "back", "movement_type": "unilateral"}))

DB results — muscle: [chest] | type: [compound]
  • Flat Barbell Bench Press (barbell, intermediate, compound)
  • Incline Dumbbell Press (dumbbell, beginner, compound)
  • Weighted Dip (bodyweight, advanced, compound)
---
DB results — muscle: [back] | type: [unilateral]
  • Single-Arm Dumbbell Row (dumbbell, beginner, unilateral)


## Part 5 — Supervisor Agent

The supervisor is the orchestrator. It:
0. Classifies the user's raw goal text into one of 7 programming paradigms — **before** calling
   `parse_inbody_text`. An unrecognized/ambiguous goal defaults to `general_fitness` rather than
   crashing or silently applying hypertrophy rules.
1. Calls `parse_inbody_text` (or `parse_inbody` for real scans) to extract structured data + flags
2. Reads the user intake form
3. Decides the training split based on days/week + experience
4. Calculates volume per muscle group using **this paradigm's** volume metric and intensity zone
   labels (with reduction factors for sleep, job type, etc. where the paradigm uses sets/week)
5. Computes the **DAY MAP** (per-day max_sets budget, using this paradigm's time-per-set values)
   and decides the muscle_group ID list — splitting "legs" into `legs_a`/`legs_b` with different
   emphasis briefs when the split has two leg days
6. Writes the session header (including the `Paradigm:` line) + DAY MAP to the shared MD memory
   file, then calls `init_plan_progress`
7. Dispatches the Exercise Recommender sub-agent once per muscle_group ID (sequential), scoping
   InBody flags per the flag-routing table, and cross-checks `get_plan_progress` after each dispatch
8. Dispatches the Plan Assembler sub-agent only once `get_plan_progress` reports all muscle groups done
9. Returns the final synthesised plan to the user

The supervisor uses `deepagents.create_deep_agent` with `AzureChatOpenAI`.


In [34]:
from deepagents import create_deep_agent

SUPERVISOR_PROMPT = """You are Tamreena's supervisor agent — the orchestrator of a personalised workout plan generation pipeline.

## Your job
0. Classify the user's raw goal text into ONE of 7 programming paradigms (see "Goal -> Paradigm
   classification" below). This MUST be your first decision, before calling parse_inbody_text.
   If the goal is ambiguous or does not match any pattern, default to general_fitness and note
   in the plan header: "Paradigm: general_fitness (original goal: '{goal text}' - defaulted to
   general fitness paradigm)". Every subsequent step uses the paradigm, not the raw goal string.
1. Call parse_inbody_text with the raw InBody text provided. Extract the structured data and FLAGS.
2. Based on the user's intake form, InBody data, and paradigm, decide:
   - Training split (Full Body / PPL / Upper-Lower / Body Part based on days_per_week + experience)
   - The exact list of muscle_group IDs you will dispatch (see "Muscle group IDs and leg-day
     differentiation" below)
   - Intensity zone per group, using the zone labels for THIS paradigm (see "Paradigm reference
     table" below) - e.g. hard/medium/soft for hypertrophy, heavy/volume/speed for strength
   - Weekly volume per group using this paradigm's volume metric (apply reduction factors below
     only where the paradigm uses a sets/week metric)
3. Compute the DAY MAP - for each training day, list its muscle_group IDs, its intensity zone
   label, and its max_sets budget, using THIS PARADIGM'S time-per-set values (see "DAY MAP budget
   formula" below). This MUST be done before any exercise-recommender is dispatched.
4. Call write_plan_memory to write the full session header: User Profile (including a
   `Paradigm: {paradigm}` line, right after Goal) + InBody Analysis + Training Plan decisions +
   the DAY MAP, using the exact DAY MAP line format given below.
5. Call init_plan_progress with session_id and the full list of muscle_group IDs from step 2.
   This must happen right after the DAY MAP is written, before any dispatch.
6. For each muscle_group ID (sequential, not parallel), call the exercise-recommender sub-agent
   using task(). Pass in the task prompt:
   - The session_id
   - The muscle_group ID (e.g. "chest", "legs_a") and the underlying muscle key to search with
     (legs_a and legs_b both search "legs")
   - Its intensity zone label and its max_sets budget from the DAY MAP (the recommender reads the
     Paradigm field itself from plan memory to know which rep/rest/RPE table applies to that zone
     - you do not need to explain the paradigm's rules yourself, just give the zone label)
   - ONLY the InBody flags relevant to this muscle_group ID (see "Flag routing" below) - never
     pass a flag the group is not supposed to receive
   - For legs_a/legs_b specifically, its distinct emphasis brief (see "Leg day differentiation")
   - Instruction to call read_plan_memory first for context, and to call mark_step_done with
     this exact muscle_group ID as its LAST action before returning
   After the task() call returns, call get_plan_progress and confirm this muscle_group ID now
   appears in "completed". If it does not, the recommender failed silently - dispatch it again
   before moving on. Do not advance to the next muscle_group ID on a mismatch.
7. Once get_plan_progress reports all_done: true for every muscle_group ID, dispatch the
   plan-assembler sub-agent using task(). Pass the session_id and training days. If all_done is
   not yet true, do NOT dispatch the assembler - go back to step 6 for whatever remains.
8. Synthesise and return the final workout plan to the user.

## Goal -> Paradigm classification (do this FIRST, before anything else)
Match the user's raw goal text against these patterns (case-insensitive, partial match is fine):
| Goal text contains...                                                | Paradigm                |
|------------------------------------------------------------------------|------------------------|
| hypertrophy / muscle / mass / bulk / size / get bigger                 | hypertrophy             |
| strength / powerlifting / 1RM / big 3 / get stronger                   | strength                |
| fat loss / weight loss / cutting / lean / burn / lose weight           | fat_loss                |
| fitness / health / general / maintain / active / wellness              | general_fitness         |
| sport / explosive / power / athletic / speed / performance             | athletic_performance    |
| endurance / marathon / running / cycling / cardio complement           | endurance_complement    |
| rehab / corrective / recovery / post-surgery / injury                  | rehabilitation          |
| anything ambiguous or unrecognized                                     | general_fitness (default - never crash) |

## Paradigm reference table (zone labels, DAY MAP time/set, volume metric)
- **hypertrophy**: zones hard/medium/soft | DAY MAP: hard 3.5 min/set, medium 2.5, mixed 3.0 |
  volume: sets/week per muscle (beginner 10-12, intermediate 14-18, advanced 18-22)
- **strength**: zones heavy/volume/speed | DAY MAP: heavy 6.0 min/set, volume 4.0, speed 3.5,
  mixed 5.0 | volume: primary-lift frequency (2-3x/week per Big 3 movement), NOT sets/week -
  name the primary movement explicitly in the DAY MAP (e.g. "barbell squat")
- **fat_loss**: zones circuit/moderate/low | DAY MAP: circuit 1.75 min/set, moderate 2.0 |
  volume: sets/week, density-optimised (beginner 10-14, intermediate 14-18, advanced 16-20)
- **general_fitness**: zone moderate only (no hard/soft split needed) | DAY MAP: 2.5 min/set
  (all sessions) | volume: movement pattern coverage - push/pull/hinge/squat/carry, >=2x/week
  each. Muscle_group IDs still apply, but frame the DAY MAP around movement pattern coverage.
  This is also the fallback paradigm for any unrecognized goal.
- **athletic_performance**: zones power/strength/conditioning | DAY MAP: power 4.0 min/set,
  conditioning 2.0, mixed 3.5 | volume: power exposure frequency + movement quality sessions/week
- **endurance_complement**: zone light only | DAY MAP: 1.75 min/set | volume: session frequency +
  external fatigue budget - reduce volume by weekly run/cycle km (<30km standard, 30-60km -20%,
  60km+ -35%). Cap leg volume conservatively. Note total weekly cardio load in the plan header.
- **rehabilitation**: zones corrective/progressive (a "hard" zone is never permitted) | DAY MAP:
  3.0 min/set (slower, controlled execution) | volume: ROM progression + movement quality, not
  sets/week. Session anchor is injury site / movement pattern, not muscle group.

## Split selection rules
- 2 days -> Full Body A/B
- 3 days + beginner -> Full Body x3
- 3 days + intermediate -> Push/Pull/Legs
- 4 days -> Upper/Lower x2 (recommended) OR Body Part 4-day for advanced
- 5 days -> PPL + Upper + Lower (advanced) OR Body Part 5-day
- 6 days -> PPL x2 (A=strength focus, B=hypertrophy focus)

## Muscle group IDs and leg-day differentiation
Use these muscle_group IDs when dispatching: chest, back, shoulders, arms, legs.
EXCEPTION: if the chosen split produces two separate leg-focused training days (this is the case
for Upper/Lower x2), do NOT dispatch "legs" once - dispatch it TWICE using two distinct IDs:
"legs_a" and "legs_b". Each needs its own task() call with a different emphasis brief:
  - legs_a brief: "Focus quad development as primary, hamstrings as secondary. Squat-pattern
    exercises first."
  - legs_b brief: "Focus hamstring and glute development as primary, quads as secondary.
    Hip-hinge pattern exercises first."
Sending the identical brief to both is a bug - they must produce different exercise selections
(e.g. squat-pattern on legs_a, RDL-pattern on legs_b). Both legs_a and legs_b should still tell
the recommender to search_rag/search_exercise_db with muscle_group="legs" (the knowledge base and
exercise DB only understand "legs") - "legs_a"/"legs_b" are only used for progress tracking
(mark_step_done) and the write_plan_memory section title.

## DAY MAP budget formula
Parse session_duration to minutes (e.g. "75min" -> 75). Subtract 10 minutes for warmup.
Divide by the minutes-per-set value for THIS PARADIGM and this day's zone (see "Paradigm reference
table" above), then floor.
Example (hypertrophy): 75min session, hard day -> floor((75-10)/3.5) = 18 max_sets.
Example (strength): 90min session, heavy day -> floor((90-10)/6.0) = 13 max_sets.

Write the DAY MAP into the shared plan memory using EXACTLY this line format (one line per day),
so downstream tools can parse it:
Day 1 - {label}: muscles [{muscle_group ids}] | max_sets: {n} | intensity: {zone label}

## Flag routing (apply BEFORE dispatching each muscle_group ID - never send a flag outside this table)
| Flag | Send to these muscle_group IDs | Never send to |
|---|---|---|
| ARM_ASYMMETRY | back, arms | chest, shoulders, legs_a, legs_b |
| LEG_ASYMMETRY | legs_a, legs_b | back, arms, chest, shoulders |
| ELEVATED_BF | all muscle_group IDs (soft hint only - see recommender's BF% rule) | - |
| TRUNK_UNDERDEVELOPED | chest, back | arms, legs_a, legs_b, shoulders |

## Volume reduction factors (apply multiplicatively, only for paradigms using a sets/week metric)
- poor sleep -> multiply by 0.8
- heavy physical job -> multiply by 0.8
- InBody asymmetry flag -> that group gets unilateral focus, NOT extra volume

Always write all decisions (including the Paradigm line and the DAY MAP) to the shared plan memory
file, and call init_plan_progress, BEFORE dispatching any sub-agent."""

SUPERVISOR_TOOLS = [
    parse_inbody_text,
    parse_inbody,
    read_plan_memory,
    write_plan_memory,
    init_plan_progress,
    get_plan_progress,
]

def build_supervisor(sub_agents):
    return create_deep_agent(
        model=llm,
        tools=SUPERVISOR_TOOLS,
        subagents=sub_agents,
        system_prompt=SUPERVISOR_PROMPT,
        name="tamreena-supervisor",
    )

print("Supervisor builder defined (v1.3 - paradigm classification).")


Supervisor builder defined (v1.3 - paradigm classification).


## Part 6 — Exercise Recommender Sub-agent

One agent definition, called once per **muscle_group ID** with a different task prompt each time. It does **not** have a different identity per muscle — the muscle group, intensity zone label, budget, and scoped flags come in via the prompt. Splits with two leg days get two dispatches (`legs_a`, `legs_b`) with different emphasis briefs, so they no longer produce identical prescriptions.

**v1.3:** the single hard/medium/soft table has been replaced with 7 paradigm-conditional tables. The agent reads the `Paradigm:` field written by the supervisor from `read_plan_memory` and applies only the matching table — it never mixes rules across paradigms.

**Process per call:**
1. `read_plan_memory` → gets full InBody data, the `Paradigm:` field, and what previous agents already wrote
2. `search_rag` → hardcoded stub returns principles + muscle-specific notes
3. `search_exercise_db` → queries the exercise DB for matching movements
4. Selects 3-5 exercises with full prescription (sets/reps/rest/RPE) from the table matching the plan's paradigm, checked against its `max_sets` budget
5. `write_plan_memory` → appends its section to the shared file
6. `mark_step_done` → records its own completion in `progress.json` as its last action
7. Returns the prescription to the supervisor


In [35]:
EXERCISE_RECOMMENDER_PROMPT = """You are Tamreena's exercise recommender. You are called once per muscle_group ID.

## What you receive from the supervisor's task prompt
- session_id
- Your muscle_group ID (e.g. "chest", "legs_a", "legs_b") - use this EXACT id for mark_step_done
  and for the write_plan_memory section_title
- The underlying muscle key to use for search_rag / search_exercise_db (for legs_a/legs_b this is
  "legs" - the knowledge base and exercise DB don't know about "legs_a"/"legs_b")
- Your intensity ZONE LABEL for this plan's paradigm (e.g. "hard" under hypertrophy, "heavy" under
  strength) and your max_sets budget for the day. The zone label alone does not tell you the
  sets/reps/rest/RPE - you must look those up in the table matching the plan's Paradigm field.
- ONLY the InBody flags relevant to you - if a flag is not explicitly given to you in the task
  prompt, do not apply it even if you notice it elsewhere in the full plan memory file
- For legs_a/legs_b specifically: a distinct emphasis brief (quad-primary vs hamstring/glute-primary)

## Your process (follow in order)
1. Call read_plan_memory with the session_id to get full context: InBody analysis, DAY MAP,
   training plan, and any previous muscle group prescriptions. Read the `Paradigm:` line from the
   User Profile section - this tells you which table in "Paradigm-conditional intensity tables"
   below to use for this entire call. Never mix rules from a different paradigm's table.
2. Call search_rag with the underlying muscle key and a query describing what you need
   (e.g. "hypertrophy chest compound movements"). If you were given an emphasis brief
   (legs_a/legs_b), reflect it in the query (e.g. "quad-dominant squat pattern exercises").
3. Call search_exercise_db with the underlying muscle key. If you were given an asymmetry flag
   for this muscle_group ID, also call with movement_type="unilateral".
4. Select 3-5 exercises based on RAG guidance and DB results, using the sets/reps/rest/RPE from
   YOUR paradigm's table for your zone label, and (if applicable) your emphasis brief. Confirm the
   total sets across all exercises does NOT exceed your max_sets budget - if it would, drop the
   lowest-priority exercise first.
5. Write the full prescription using write_plan_memory, with section_title = your exact
   muscle_group ID + zone label (e.g. "legs_a - medium", not "legs - medium").
6. Call mark_step_done with session_id and your exact muscle_group ID as your LAST tool call,
   before returning your summary to the supervisor.
7. Return the prescription summary to the supervisor.

## Paradigm-conditional intensity tables - use ONLY the table matching the plan's Paradigm field

### Paradigm: hypertrophy
| Zone   | Sets | Reps  | Rest    | RPE | Focus |
|--------|------|-------|---------|-----|-------|
| hard   | 4-5  | 6-8   | 2-3 min | 8-9 | heavy compound first |
| medium | 3-4  | 10-12 | 90s     | 7   | compound + isolation |
| soft   | 3    | 15+   | 60s     | 5-6 | corrective / unilateral |

### Paradigm: strength
| Zone   | Sets | Reps       | Rest    | RPE  | Focus |
|--------|------|------------|---------|------|-------|
| heavy  | 4-5  | 1-5 main   | 4-8 min | 9+   | primary movement anchor - name it explicitly |
| volume | 3-5  | 3-8 suppl  | 3-5 min | 7-8  | same movement pattern, reduced load |
| speed  | 3-4  | 2-4        | 3 min   | 6-7  | 60-70% 1RM, technique / bar speed focus |
Rep windows: main 1-5 / supplemental 3-8 / accessories 6-12. Name the primary movement explicitly
in your write-up (e.g. "Barbell Back Squat", not just "squat variation").

### Paradigm: fat_loss
| Zone     | Sets | Reps  | Rest   | RPE | Focus |
|----------|------|-------|--------|-----|-------|
| circuit  | 3-4  | 15-20 | 45s    | 7-8 | density, superset-friendly |
| moderate | 3    | 12-15 | 60-75s | 6-7 | compound movements, full ROM |
| low      | 2-3  | 15+   | 45s    | 5-6 | corrective / finisher |

### Paradigm: general_fitness
| Zone     | Sets | Reps | Rest    | RPE | Focus |
|----------|------|------|---------|-----|-------|
| moderate | 3-4  | 8-15 | 60-120s | 6-8 | movement pattern coverage |
Anchor your selection on movement pattern (push/pull/hinge/squat/carry) rather than isolating
the muscle_group ID alone. All days use "moderate" - there is no hard/soft distinction here.

### Paradigm: athletic_performance
| Zone         | Sets | Reps  | Rest    | RPE | Focus |
|--------------|------|-------|---------|-----|-------|
| power        | 3-5  | 3-6   | 2-4 min | 8-9 | explosive, sport-relevant movement |
| strength     | 3-4  | 5-10  | 2-3 min | 7-8 | compound patterns |
| conditioning | 3-4  | 12-20 | 60-90s  | 6-7 | muscular endurance, sport carry-over |
Prioritise multi-joint, sport-transferable movements over isolation work.

### Paradigm: endurance_complement
| Zone  | Sets | Reps  | Rest   | RPE | Focus |
|-------|------|-------|--------|-----|-------|
| light | 2-3  | 15-25 | 30-60s | 5-7 | muscular endurance, no heavy loading |
Only the "light" zone exists for this paradigm - if you receive any other zone label on an
endurance_complement plan, treat it as "light". Keep leg volume conservative regardless of your
max_sets budget - running/cycling already loads legs; do not fill the full budget just because
it is available.

### Paradigm: rehabilitation
| Zone        | Sets | Reps  | Rest    | RPE | Focus |
|-------------|------|-------|---------|-----|-------|
| corrective  | 2-3  | 10-15 | 60-90s  | 4-6 | pain-free ROM only, slow tempo |
| progressive | 3    | 10-20 | 60-90s  | 5-7 | gradual load increase, movement quality |
No "hard" zone exists for this paradigm. Every exercise must be checked against the user's stated
injuries - pain-free range of motion is the primary constraint, load is secondary.

## Asymmetry rule
If you were explicitly given an asymmetry flag for THIS muscle_group ID -> at least one exercise
MUST be unilateral. Always note "start on weaker side." If no asymmetry flag was given to you,
do not apply unilateral prescription on your own initiative.

## Elevated BF% rule (soft hint only - do not treat as a fixed override)
If you were given ELEVATED_BF: prefer the upper end of your zone's rep range (e.g. 8 over 6 for a
hypertrophy "hard" zone, 12 over 10 for "medium") as a general lean, not a fixed rule applied
identically to every exercise - reps should still vary across your exercises based on exercise
type and load. This flag is most relevant for hypertrophy and fat_loss paradigms; for paradigms
whose rep windows are already fixed by injury or protocol (rehabilitation, endurance_complement),
do not let it override the table above.

## Output format for write_plan_memory (section_title = "{your muscle_group ID} - {ZONE LABEL}")
```
1. Exercise Name   {sets}x{reps} | Rest {time} | RPE {n}
   -> why this exercise / key technique cue
2. ...
Evidence: [1 sentence from RAG supporting this selection]
```"""

EXERCISE_RECOMMENDER = {
    "model": llm,
    "tools": [read_plan_memory, write_plan_memory, search_rag, search_exercise_db, mark_step_done],
    "system_prompt": EXERCISE_RECOMMENDER_PROMPT,
    "name": "exercise-recommender",
    "description": "Recommends 3-5 exercises with full prescription (sets/reps/rest/RPE) for a single muscle_group ID, using the intensity table matching the plan's paradigm, RAG guidance, and the exercise DB. Marks its own completion via mark_step_done."
}

print("Exercise Recommender sub-agent defined (v1.3 - 7 paradigm-conditional tables).")


Exercise Recommender sub-agent defined (v1.3 - 7 paradigm-conditional tables).


## Part 7 — Plan Assembler Sub-agent

Called once, after the supervisor confirms (via `get_plan_progress`) that every muscle_group ID has been marked done. It reads everything the exercise recommender agents wrote and arranges it into a coherent weekly schedule.

**Rules it applies:**
- Refuses to run (`validate_plan_completeness`) if any expected muscle group is missing — this is the check that would have caught the earlier bug where "back" was silently dropped
- Never train the same muscle group on consecutive days
- Hardest session placed where recovery time is longest after it
- Upper/Lower splits alternate upper and lower days — `legs_a`/`legs_b` are separate sections with distinct content, so the two leg days are no longer identical
- Each session gets a warm-up note specific to its muscle group focus
- Enforces the DAY MAP's `max_sets` budget per day via `validate_session_duration`, trimming if needed
- Calculates total weekly volume per muscle group and flags if under/over target

In [36]:
PLAN_ASSEMBLER_PROMPT = """You are Tamreena's plan assembler. You are called once after all muscle_group ID prescriptions are complete.

## Your process (follow in order)
0. Call validate_plan_completeness with the session_id FIRST, before doing any scheduling work.
   If it returns INCOMPLETE, STOP — report exactly which muscle_group IDs are missing back to
   the supervisor instead of assembling a plan around a gap. Do not guess or fill in a missing
   muscle group yourself.
1. Call read_plan_memory with the session_id to get all muscle group prescriptions, the DAY MAP,
   and the split type.
2. Arrange the prescriptions into a weekly schedule following the rules below. Read the max_sets
   budget for each day from the DAY MAP.
3. Call write_plan_memory with section_title="Weekly Schedule" to save the final plan.
4. Call validate_session_duration with the session_id. If it returns VIOLATIONS, trim in this
   order (lowest priority first) and re-check until it returns PASS:
   a. Isolation exercises that duplicate stimulus already in the session — keep the one with
      the stronger RAG justification.
   b. Accessory/corrective exercises beyond the first per session.
   c. Additional sets from the lowest-RPE exercise in the day.
   Never remove: the primary compound lift for any muscle group, or any unilateral exercise
   flagged for asymmetry correction.
5. Return the full formatted weekly plan to the supervisor.

## Scheduling rules (mandatory)
- NEVER place the same muscle group on consecutive days.
- Place the hardest session (most volume / heaviest loading) where the longest recovery window
  follows it (e.g. before a rest day).
- For Upper/Lower splits: alternate Upper → Lower → Upper → Lower. legs_a and legs_b are two
  DIFFERENT sections in plan memory with different exercises — schedule each on its own Lower
  day using its own content. Do not copy one leg day's exercises onto the other.
- For PPL: Push → Pull → Legs in order, repeat if 6 days.

## Session format
For each training day output:

### Day {N} — {DayName}: {Session Focus}
**Warm-up:** {2-sentence specific warm-up for this session's muscle focus}

| # | Exercise | Sets × Reps | Rest | RPE |
|---|----------|-------------|------|-----|
| 1 | ...      | ...         | ...  | ... |

IMPORTANT: in every table row, sets and reps MUST be separated by the × character (e.g. "4×12").
Never merge them into a single number like "412" — validate_session_duration cannot count sets
correctly if the × is missing.

**Coaching notes:** {1 key tip for this session}

---

## End of plan output
After all days, add:

### Weekly Volume Summary
| Muscle Group | Sets/Week | Target | Status |
|---|---|---|---|
| chest | X | 14-18 | met / under / over |
...
If any muscle group shows 0 sets/week here, that is a hard failure, not just a status note — it
means step 0's validate_plan_completeness should have caught a missing prescription. Re-run step 0
rather than reporting a plan with a 0-set muscle group.

### Recovery Notes
- {any asymmetry corrections to remind the user of}
- {any BF% or sleep-based adjustments made}"""

PLAN_ASSEMBLER = {
    "model": llm,
    "tools": [read_plan_memory, write_plan_memory, validate_plan_completeness, validate_session_duration],
    "system_prompt": PLAN_ASSEMBLER_PROMPT,
    "name": "plan-assembler",
    "description": "Arranges all completed muscle-group prescriptions into a full weekly training schedule, following split and recovery rules. Refuses to run if muscle groups are missing, and enforces the session-duration budget from the DAY MAP."
}

print("Plan Assembler sub-agent defined.")


Plan Assembler sub-agent defined.


## Part 8 — End-to-End Pipeline Run

Each of the 13 cells below is one self-contained test case pulled directly from
`tests/test_cases.py` — same intake-form text, same InBody text, same validation
checklist the test file documents. Every cell can be run on its own (after the
setup cells above have run once) since each one imports its own case data, builds
a fresh supervisor, and uses its own `SESSION_ID`.

| Case | Focus | Expected RAG triggers |
|---|---|---|
| 01 | Baseline hypertrophy, clean path | none |
| 02 | Beginner female, fat loss, 3 days | none |
| 03 | Advanced strength / powerlifting, 5 days | none |
| 04 | Two simultaneous injuries (shoulder + knee) | T1, T4 |
| 05 | Severe arm + leg asymmetry | T2 |
| 06 | Obese beginner, high BF% | T5, T6 |
| 07 | Very lean advanced athlete | T5 |
| 08 | Older adult, joint limitations | T1, T6 |
| 09 | Time-compressed (5 days x 40min) | none |
| 10 | Minimal home equipment | none |
| 11 | Multi-flag combination (hardest case) | T1, T2, T3, T5, T7 |
| 12 | Body recomposition goal | none |
| 13 | Endurance athlete adding strength (concurrent training) | T4 |

Note: `tests/test_cases.py` documents RAG triggers (T1-T7) and a nutrition agent —
those describe the fuller system vision. This notebook's pipeline (Supervisor +
Exercise Recommender + Plan Assembler, hardcoded RAG stub, no nutrition agent yet)
does not implement RAG-trigger routing or nutrition coordination, so those parts of
each case's validation checklist won't apply yet — run the cells to see how today's
paradigm-classification pipeline actually handles each scenario, and use the printed
checklist as a manual read of what still needs building.

After any cell runs, open `sessions/{session_id}/plan.md` to inspect the shared
memory file, or use the inspection cells at the very end (they operate on whichever
`SESSION_ID` was set by the last case cell you ran).


In [37]:
import uuid
from datetime import datetime
from tests.test_cases import CASE_01_INTAKE, CASE_01_INBODY, CASE_01_META

# -- CASE 01 -- BASELINE (clean path) ------------------------------------------------
# Clean intermediate hypertrophy — no triggers, validates happy path
# Expected RAG triggers: NONE
USER_INTAKE = CASE_01_INTAKE
SAMPLE_INBODY = CASE_01_INBODY

SESSION_ID = str(uuid.uuid4())
print(f"Session ({CASE_01_META['id']}): {SESSION_ID}")

supervisor = build_supervisor(sub_agents=[EXERCISE_RECOMMENDER, PLAN_ASSEMBLER])

user_message = f"""SESSION_ID: {SESSION_ID}

{USER_INTAKE}

INBODY RAW TEXT:
{SAMPLE_INBODY}

Generate a full personalised workout plan for this user."""

print(f"\nStarting pipeline at {datetime.now().strftime('%H:%M:%S')}...\n")

result = supervisor.invoke(
    {"messages": [{"role": "user", "content": user_message}]},
    print_mode="updates",
)

final_plan = result["messages"][-1].content
print("\n" + "="*60)
print("FINAL PLAN OUTPUT")
print("="*60)
print(final_plan)
print(f"\nSession file: sessions/{SESSION_ID}/plan.md")

print("\n" + "-"*60)
print(f"Manual validation checklist -- {CASE_01_META['id']}")
print("-"*60)
for item in CASE_01_META["validate"]:
    print(f"  [ ] {item}")


Session (case_01_baseline): 983b3237-3432-42cb-8da3-183268945c96

Starting pipeline at 16:52:51...

[updates] {'PatchToolCallsMiddleware.before_agent': None}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 173, 'prompt_tokens': 8405, 'total_tokens': 8578, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_ff76eaacab', 'id': 'chatcmpl-DzjTLvwNtnxD4sv35djR95MmYlUYh', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, name='tamreena-supervisor', id='lc_run--019f4727-1612-7e23-aba1-76a2328a634c-0', tool_calls=[{'name': 'parse_inbody_text', 'args': {'session_id': '983b3237-3432-42cb-8da3-183268945c96', 'raw_text': 'InBod

In [ ]:
import uuid
from datetime import datetime
from tests.test_cases import CASE_02_INTAKE, CASE_02_INBODY, CASE_02_META

# -- CASE 02 -- BEGINNER FEMALE, FAT LOSS ------------------------------------------------
# Beginner female, fat loss, 3 days — validates goal-specific split and beginner volume caps
# Expected RAG triggers: NONE
USER_INTAKE = CASE_02_INTAKE
SAMPLE_INBODY = CASE_02_INBODY

SESSION_ID = str(uuid.uuid4())
print(f"Session ({CASE_02_META['id']}): {SESSION_ID}")

supervisor = build_supervisor(sub_agents=[EXERCISE_RECOMMENDER, PLAN_ASSEMBLER])

user_message = f"""SESSION_ID: {SESSION_ID}

{USER_INTAKE}

INBODY RAW TEXT:
{SAMPLE_INBODY}

Generate a full personalised workout plan for this user."""

print(f"\nStarting pipeline at {datetime.now().strftime('%H:%M:%S')}...\n")

result = supervisor.invoke(
    {"messages": [{"role": "user", "content": user_message}]},
    print_mode="updates",
)

final_plan = result["messages"][-1].content
print("\n" + "="*60)
print("FINAL PLAN OUTPUT")
print("="*60)
print(final_plan)
print(f"\nSession file: sessions/{SESSION_ID}/plan.md")

print("\n" + "-"*60)
print(f"Manual validation checklist -- {CASE_02_META['id']}")
print("-"*60)
for item in CASE_02_META["validate"]:
    print(f"  [ ] {item}")


In [38]:
import uuid
from datetime import datetime
from tests.test_cases import CASE_03_INTAKE, CASE_03_INBODY, CASE_03_META

# -- CASE 03 -- ADVANCED STRENGTH / POWERLIFTING ------------------------------------------------
# Advanced powerlifter, 5 days — validates strength periodization, compound-first logic
# Expected RAG triggers: NONE
USER_INTAKE = CASE_03_INTAKE
SAMPLE_INBODY = CASE_03_INBODY

SESSION_ID = str(uuid.uuid4())
print(f"Session ({CASE_03_META['id']}): {SESSION_ID}")

supervisor = build_supervisor(sub_agents=[EXERCISE_RECOMMENDER, PLAN_ASSEMBLER])

user_message = f"""SESSION_ID: {SESSION_ID}

{USER_INTAKE}

INBODY RAW TEXT:
{SAMPLE_INBODY}

Generate a full personalised workout plan for this user."""

print(f"\nStarting pipeline at {datetime.now().strftime('%H:%M:%S')}...\n")

result = supervisor.invoke(
    {"messages": [{"role": "user", "content": user_message}]},
    print_mode="updates",
)

final_plan = result["messages"][-1].content
print("\n" + "="*60)
print("FINAL PLAN OUTPUT")
print("="*60)
print(final_plan)
print(f"\nSession file: sessions/{SESSION_ID}/plan.md")

print("\n" + "-"*60)
print(f"Manual validation checklist -- {CASE_03_META['id']}")
print("-"*60)
for item in CASE_03_META["validate"]:
    print(f"  [ ] {item}")

Session (case_03_advanced_strength): c9caa385-bb1a-4bf2-9d4b-7581e5e6445a

Starting pipeline at 16:57:22...

[updates] {'PatchToolCallsMiddleware.before_agent': None}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 178, 'prompt_tokens': 8410, 'total_tokens': 8588, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 8064}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_ff76eaacab', 'id': 'chatcmpl-DzjXiD5d7uYHm1P6x0LcxcVTuwnck', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, name='tamreena-supervisor', id='lc_run--019f472b-3969-7e73-8c6f-66d34632cdc1-0', tool_calls=[{'name': 'parse_inbody_text', 'args': {'session_id': 'c9caa385-bb1a-4bf2-9d4b-7581e5e6445a', 'raw_t

In [ ]:
import uuid
from datetime import datetime
from tests.test_cases import CASE_04_INTAKE, CASE_04_INBODY, CASE_04_META

# -- CASE 04 -- MULTIPLE ACTIVE INJURIES ------------------------------------------------
# Two simultaneous injuries (shoulder + knee) that create conflicting exercise constraints
# Expected RAG triggers: T1_injury_pain_flag, T4_conflicting_contraindications
USER_INTAKE = CASE_04_INTAKE
SAMPLE_INBODY = CASE_04_INBODY

SESSION_ID = str(uuid.uuid4())
print(f"Session ({CASE_04_META['id']}): {SESSION_ID}")

supervisor = build_supervisor(sub_agents=[EXERCISE_RECOMMENDER, PLAN_ASSEMBLER])

user_message = f"""SESSION_ID: {SESSION_ID}


{USER_INTAKE}

INBODY RAW TEXT:
{SAMPLE_INBODY}

Generate a full personalised workout plan for this user."""

print(f"\nStarting pipeline at {datetime.now().strftime('%H:%M:%S')}...\n")

result = supervisor.invoke(
    {"messages": [{"role": "user", "content": user_message}]},
    print_mode="updates",
)

final_plan = result["messages"][-1].content
print("\n" + "="*60)
print("FINAL PLAN OUTPUT")
print("="*60)
print(final_plan)
print(f"\nSession file: sessions/{SESSION_ID}/plan.md")

print("\n" + "-"*60)
print(f"Manual validation checklist -- {CASE_04_META['id']}")
print("-"*60)
for item in CASE_04_META["validate"]:
    print(f"  [ ] {item}")


In [ ]:
import uuid
from datetime import datetime
from tests.test_cases import CASE_05_INTAKE, CASE_05_INBODY, CASE_05_META

# -- CASE 05 -- SEVERE SEGMENTAL IMBALANCE ------------------------------------------------
# Severe arm asymmetry (1.58kg gap) + notable leg asymmetry — RAG must fire for imbalance
# Expected RAG triggers: T2_muscle_imbalance_detected
USER_INTAKE = CASE_05_INTAKE
SAMPLE_INBODY = CASE_05_INBODY

SESSION_ID = str(uuid.uuid4())
print(f"Session ({CASE_05_META['id']}): {SESSION_ID}")

supervisor = build_supervisor(sub_agents=[EXERCISE_RECOMMENDER, PLAN_ASSEMBLER])

user_message = f"""SESSION_ID: {SESSION_ID}

{USER_INTAKE}

INBODY RAW TEXT:
{SAMPLE_INBODY}

Generate a full personalised workout plan for this user."""

print(f"\nStarting pipeline at {datetime.now().strftime('%H:%M:%S')}...\n")

result = supervisor.invoke(
    {"messages": [{"role": "user", "content": user_message}]},
    print_mode="updates",
)

final_plan = result["messages"][-1].content
print("\n" + "="*60)
print("FINAL PLAN OUTPUT")
print("="*60)
print(final_plan)
print(f"\nSession file: sessions/{SESSION_ID}/plan.md")

print("\n" + "-"*60)
print(f"Manual validation checklist -- {CASE_05_META['id']}")
print("-"*60)
for item in CASE_05_META["validate"]:
    print(f"  [ ] {item}")


In [ ]:
import uuid
from datetime import datetime
from tests.test_cases import CASE_06_INTAKE, CASE_06_INBODY, CASE_06_META

# -- CASE 06 -- OBESE BEGINNER, HIGH BF% ------------------------------------------------
# High BF% (41.3%) + high body weight + beginner — extreme InBody + special population flags
# Expected RAG triggers: T5_extreme_inbody_values, T6_special_population_flag
USER_INTAKE = CASE_06_INTAKE
SAMPLE_INBODY = CASE_06_INBODY

SESSION_ID = str(uuid.uuid4())
print(f"Session ({CASE_06_META['id']}): {SESSION_ID}")

supervisor = build_supervisor(sub_agents=[EXERCISE_RECOMMENDER, PLAN_ASSEMBLER])

user_message = f"""SESSION_ID: {SESSION_ID}

{USER_INTAKE}

INBODY RAW TEXT:
{SAMPLE_INBODY}

Generate a full personalised workout plan for this user."""

print(f"\nStarting pipeline at {datetime.now().strftime('%H:%M:%S')}...\n")

result = supervisor.invoke(
    {"messages": [{"role": "user", "content": user_message}]},
    print_mode="updates",
)

final_plan = result["messages"][-1].content
print("\n" + "="*60)
print("FINAL PLAN OUTPUT")
print("="*60)
print(final_plan)
print(f"\nSession file: sessions/{SESSION_ID}/plan.md")

print("\n" + "-"*60)
print(f"Manual validation checklist -- {CASE_06_META['id']}")
print("-"*60)
for item in CASE_06_META["validate"]:
    print(f"  [ ] {item}")


In [ ]:
import uuid
from datetime import datetime
from tests.test_cases import CASE_07_INTAKE, CASE_07_INBODY, CASE_07_META

# -- CASE 07 -- VERY LEAN ATHLETE, MUSCLE BUILDING ------------------------------------------------
# Very low BF% (6.8%) athlete — T5 fires, nutrition agent coordination critical
# Expected RAG triggers: T5_extreme_inbody_values
USER_INTAKE = CASE_07_INTAKE
SAMPLE_INBODY = CASE_07_INBODY

SESSION_ID = str(uuid.uuid4())
print(f"Session ({CASE_07_META['id']}): {SESSION_ID}")

supervisor = build_supervisor(sub_agents=[EXERCISE_RECOMMENDER, PLAN_ASSEMBLER])

user_message = f"""SESSION_ID: {SESSION_ID}

{USER_INTAKE}

INBODY RAW TEXT:
{SAMPLE_INBODY}

Generate a full personalised workout plan for this user."""

print(f"\nStarting pipeline at {datetime.now().strftime('%H:%M:%S')}...\n")

result = supervisor.invoke(
    {"messages": [{"role": "user", "content": user_message}]},
    print_mode="updates",
)

final_plan = result["messages"][-1].content
print("\n" + "="*60)
print("FINAL PLAN OUTPUT")
print("="*60)
print(final_plan)
print(f"\nSession file: sessions/{SESSION_ID}/plan.md")

print("\n" + "-"*60)
print(f"Manual validation checklist -- {CASE_07_META['id']}")
print("-"*60)
for item in CASE_07_META["validate"]:
    print(f"  [ ] {item}")


In [ ]:
import uuid
from datetime import datetime
from tests.test_cases import CASE_08_INTAKE, CASE_08_INBODY, CASE_08_META

# -- CASE 08 -- OLDER ADULT, JOINT LIMITATIONS ------------------------------------------------
# 52yr old with bilateral knee arthritis + L4-L5 — T1 + T6 (special population)
# Expected RAG triggers: T1_injury_pain_flag, T6_special_population_flag
USER_INTAKE = CASE_08_INTAKE
SAMPLE_INBODY = CASE_08_INBODY

SESSION_ID = str(uuid.uuid4())
print(f"Session ({CASE_08_META['id']}): {SESSION_ID}")

supervisor = build_supervisor(sub_agents=[EXERCISE_RECOMMENDER, PLAN_ASSEMBLER])

user_message = f"""SESSION_ID: {SESSION_ID}

{USER_INTAKE}

INBODY RAW TEXT:
{SAMPLE_INBODY}

Generate a full personalised workout plan for this user."""

print(f"\nStarting pipeline at {datetime.now().strftime('%H:%M:%S')}...\n")

result = supervisor.invoke(
    {"messages": [{"role": "user", "content": user_message}]},
    print_mode="updates",
)

final_plan = result["messages"][-1].content
print("\n" + "="*60)
print("FINAL PLAN OUTPUT")
print("="*60)
print(final_plan)
print(f"\nSession file: sessions/{SESSION_ID}/plan.md")

print("\n" + "-"*60)
print(f"Manual validation checklist -- {CASE_08_META['id']}")
print("-"*60)
for item in CASE_08_META["validate"]:
    print(f"  [ ] {item}")


In [ ]:
import uuid
from datetime import datetime
from tests.test_cases import CASE_09_INTAKE, CASE_09_INBODY, CASE_09_META

# -- CASE 09 -- TIME-COMPRESSED INTERMEDIATE ------------------------------------------------
# 5 days x 40min — volume compression logic, dense session design
# Expected RAG triggers: NONE
USER_INTAKE = CASE_09_INTAKE
SAMPLE_INBODY = CASE_09_INBODY

SESSION_ID = str(uuid.uuid4())
print(f"Session ({CASE_09_META['id']}): {SESSION_ID}")

supervisor = build_supervisor(sub_agents=[EXERCISE_RECOMMENDER, PLAN_ASSEMBLER])

user_message = f"""SESSION_ID: {SESSION_ID}

{USER_INTAKE}

INBODY RAW TEXT:
{SAMPLE_INBODY}

Generate a full personalised workout plan for this user."""

print(f"\nStarting pipeline at {datetime.now().strftime('%H:%M:%S')}...\n")

result = supervisor.invoke(
    {"messages": [{"role": "user", "content": user_message}]},
    print_mode="updates",
)

final_plan = result["messages"][-1].content
print("\n" + "="*60)
print("FINAL PLAN OUTPUT")
print("="*60)
print(final_plan)
print(f"\nSession file: sessions/{SESSION_ID}/plan.md")

print("\n" + "-"*60)
print(f"Manual validation checklist -- {CASE_09_META['id']}")
print("-"*60)
for item in CASE_09_META["validate"]:
    print(f"  [ ] {item}")


In [ ]:
import uuid
from datetime import datetime
from tests.test_cases import CASE_10_INTAKE, CASE_10_INBODY, CASE_10_META

# -- CASE 10 -- MINIMAL HOME EQUIPMENT ------------------------------------------------
# Home gym with limited equipment — tests exercise filter fallback chain
# Expected RAG triggers: NONE
USER_INTAKE = CASE_10_INTAKE
SAMPLE_INBODY = CASE_10_INBODY

SESSION_ID = str(uuid.uuid4())
print(f"Session ({CASE_10_META['id']}): {SESSION_ID}")

supervisor = build_supervisor(sub_agents=[EXERCISE_RECOMMENDER, PLAN_ASSEMBLER])

user_message = f"""SESSION_ID: {SESSION_ID}

{USER_INTAKE}

INBODY RAW TEXT:
{SAMPLE_INBODY}

Generate a full personalised workout plan for this user."""

print(f"\nStarting pipeline at {datetime.now().strftime('%H:%M:%S')}...\n")

result = supervisor.invoke(
    {"messages": [{"role": "user", "content": user_message}]},
    print_mode="updates",
)

final_plan = result["messages"][-1].content
print("\n" + "="*60)
print("FINAL PLAN OUTPUT")
print("="*60)
print(final_plan)
print(f"\nSession file: sessions/{SESSION_ID}/plan.md")

print("\n" + "-"*60)
print(f"Manual validation checklist -- {CASE_10_META['id']}")
print("-"*60)
for item in CASE_10_META["validate"]:
    print(f"  [ ] {item}")


In [ ]:
import uuid
from datetime import datetime
from tests.test_cases import CASE_11_INTAKE, CASE_11_INBODY, CASE_11_META

# -- CASE 11 -- MULTI-FLAG COMBINATION (hardest case) ------------------------------------------------
# 4 simultaneous RAG flags: injury (T1) + imbalance (T2) + plateau (T3) + extreme BF (T5) → T7
# Expected RAG triggers: T1_injury_pain_flag, T2_muscle_imbalance_detected, T3_plateau_detected, T5_extreme_inbody_values, T7_multi_flag_combination
USER_INTAKE = CASE_11_INTAKE
SAMPLE_INBODY = CASE_11_INBODY

SESSION_ID = str(uuid.uuid4())
print(f"Session ({CASE_11_META['id']}): {SESSION_ID}")

supervisor = build_supervisor(sub_agents=[EXERCISE_RECOMMENDER, PLAN_ASSEMBLER])

user_message = f"""SESSION_ID: {SESSION_ID}

{USER_INTAKE}

INBODY RAW TEXT:
{SAMPLE_INBODY}

Generate a full personalised workout plan for this user."""

print(f"\nStarting pipeline at {datetime.now().strftime('%H:%M:%S')}...\n")

result = supervisor.invoke(
    {"messages": [{"role": "user", "content": user_message}]},
    print_mode="updates",
)

final_plan = result["messages"][-1].content
print("\n" + "="*60)
print("FINAL PLAN OUTPUT")
print("="*60)
print(final_plan)
print(f"\nSession file: sessions/{SESSION_ID}/plan.md")

print("\n" + "-"*60)
print(f"Manual validation checklist -- {CASE_11_META['id']}")
print("-"*60)
for item in CASE_11_META["validate"]:
    print(f"  [ ] {item}")


In [ ]:
import uuid
from datetime import datetime
from tests.test_cases import CASE_12_INTAKE, CASE_12_INBODY, CASE_12_META

# -- CASE 12 -- RECOMPOSITION GOAL (fat loss + muscle gain) ------------------------------------------------
# Recomp goal (unique nutrition/training crossover) — validates agent coordination on dual objective
# Expected RAG triggers: NONE
USER_INTAKE = CASE_12_INTAKE
SAMPLE_INBODY = CASE_12_INBODY

SESSION_ID = str(uuid.uuid4())
print(f"Session ({CASE_12_META['id']}): {SESSION_ID}")

supervisor = build_supervisor(sub_agents=[EXERCISE_RECOMMENDER, PLAN_ASSEMBLER])

user_message = f"""SESSION_ID: {SESSION_ID}

{USER_INTAKE}

INBODY RAW TEXT:
{SAMPLE_INBODY}

Generate a full personalised workout plan for this user."""

print(f"\nStarting pipeline at {datetime.now().strftime('%H:%M:%S')}...\n")

result = supervisor.invoke(
    {"messages": [{"role": "user", "content": user_message}]},
    print_mode="updates",
)

final_plan = result["messages"][-1].content
print("\n" + "="*60)
print("FINAL PLAN OUTPUT")
print("="*60)
print(final_plan)
print(f"\nSession file: sessions/{SESSION_ID}/plan.md")

print("\n" + "-"*60)
print(f"Manual validation checklist -- {CASE_12_META['id']}")
print("-"*60)
for item in CASE_12_META["validate"]:
    print(f"  [ ] {item}")


In [ ]:
import uuid
from datetime import datetime
from tests.test_cases import CASE_13_INTAKE, CASE_13_INBODY, CASE_13_META

# -- CASE 13 -- ENDURANCE ATHLETE ADDING STRENGTH ------------------------------------------------
# Concurrent training scenario — running + lifting, agents must account for total fatigue
# Expected RAG triggers: T4_conflicting_contraindications
USER_INTAKE = CASE_13_INTAKE
SAMPLE_INBODY = CASE_13_INBODY

SESSION_ID = str(uuid.uuid4())
print(f"Session ({CASE_13_META['id']}): {SESSION_ID}")

supervisor = build_supervisor(sub_agents=[EXERCISE_RECOMMENDER, PLAN_ASSEMBLER])

user_message = f"""SESSION_ID: {SESSION_ID}

{USER_INTAKE}

INBODY RAW TEXT:
{SAMPLE_INBODY}

Generate a full personalised workout plan for this user."""

print(f"\nStarting pipeline at {datetime.now().strftime('%H:%M:%S')}...\n")

result = supervisor.invoke(
    {"messages": [{"role": "user", "content": user_message}]},
    print_mode="updates",
)

final_plan = result["messages"][-1].content
print("\n" + "="*60)
print("FINAL PLAN OUTPUT")
print("="*60)
print(final_plan)
print(f"\nSession file: sessions/{SESSION_ID}/plan.md")

print("\n" + "-"*60)
print(f"Manual validation checklist -- {CASE_13_META['id']}")
print("-"*60)
for item in CASE_13_META["validate"]:
    print(f"  [ ] {item}")


In [ ]:
# Inspect the shared memory file for whichever case you ran last
# — shows every agent's output in order for that SESSION_ID
session_file = os.path.join("..", "sessions", SESSION_ID, "plan.md")
if os.path.exists(session_file):
    with open(session_file, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("Session file not found — pipeline may not have written to memory yet.")


In [ ]:
# -- Verify progress tracking + session-duration budget for whichever case you ran last --
# This is the check that would have caught the earlier bug where "back" was
# silently dropped with 0 sets scheduled and no error raised.

progress_file = os.path.join("..", "sessions", SESSION_ID, "progress.json")
if os.path.exists(progress_file):
    with open(progress_file, "r", encoding="utf-8") as f:
        print("progress.json:")
        print(json.dumps(json.load(f), indent=2))
else:
    print("progress.json not found — init_plan_progress may not have been called.")

print("\nvalidate_plan_completeness result:")
print(validate_plan_completeness.invoke({"session_id": SESSION_ID}))

print("\nvalidate_session_duration result:")
print(validate_session_duration.invoke({"session_id": SESSION_ID}))


In [ ]:
# -- Show just the Plan Assembler's output for whichever case you ran last --
# Everything above "## Weekly Schedule" in plan.md is each exercise-recommender's
# raw per-muscle working notes (audit trail). This section is the one written by
# the Plan Assembler (write_plan_memory, section_title="Weekly Schedule") — it's
# the part that actually combines pieces from every muscle group into Day 1-N.

session_file = os.path.join("..", "sessions", SESSION_ID, "plan.md")
if os.path.exists(session_file):
    with open(session_file, "r", encoding="utf-8") as f:
        content = f.read()
    marker = "## Weekly Schedule"
    if marker in content:
        print(content[content.index(marker):])
    else:
        print("No '## Weekly Schedule' section found yet — Plan Assembler may not have run.")
else:
    print("Session file not found.")
